# MLflow Walkthrough

This notebook demonstrates local MLflow tracking for the housing workflow. It records parameters, metrics, artifacts, and parent/child run relationships.

Start the UI from the project root with `mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000`, then execute this notebook cell by cell.

In [ ]:
from pathlib import Path
import mlflow

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'setup.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').resolve()}"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment('housing-project-walkthrough')
mlflow.get_tracking_uri()

## Basic run: parameters, metrics, and artifacts

In [ ]:
with mlflow.start_run(run_name='basic-demo') as run:
    mlflow.log_param('model_type', 'random_forest')
    mlflow.log_param('n_estimators', 30)
    mlflow.log_metric('validation_rmse', 48133.59)
    mlflow.log_metric('validation_mae', 31620.47)
    mlflow.log_text('Parameters and metrics were recorded locally.', 'notes.txt')
    run.info.run_id

## Nested runs

A parent run represents one complete workflow. Each stage gets a nested child run, which makes the UI show both the overall execution and its individual tasks.

In [ ]:
with mlflow.start_run(run_name='housing-workflow-demo') as parent:
    mlflow.log_param('workflow', 'ingest-train-score')
    with mlflow.start_run(run_name='data-ingestion', nested=True) as ingestion:
        mlflow.log_param('split_strategy', 'stratified_income_category')
        mlflow.log_metric('train_rows', 16512)
    with mlflow.start_run(run_name='model-training', nested=True) as training:
        mlflow.log_param('regressor', 'RandomForestRegressor')
        mlflow.log_param('n_estimators', 30)
    with mlflow.start_run(run_name='model-scoring', nested=True) as scoring:
        mlflow.log_metric('rmse', 48133.59)
        mlflow.log_metric('mae', 31620.47)
    parent.info.run_id

## Run the project workflow

From a terminal at the project root, run `python scripts/run_workflow.py`. Refresh the MLflow UI and inspect the parent run, nested child runs, logged parameters, metrics, and artifacts.